# Extending NeuRosetta

Patterns for custom analysis without forking the library. Read
{doc}`architecture` first for the layer model (`utils` → `ops` → `api`).

This notebook shows **user-level extension**: attach analysis results as graph
properties and reuse the functional API.


In [1]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import neurosetta as nr

%matplotlib inline

EXAMPLE_ID = 720575940596125868
tree = nr.load_example_data(EXAMPLE_ID)
forest = nr.load_example_data()
print(tree)
print(f"Forest ids: {forest.ids()}")


Tree(ID=720575940596125868) with 2010 nodes
Forest ids: [720575940596125868, 720575940599459782, 720575940599704006, 720575940599729862]


## Attach a custom vertex property


In [2]:
import numpy as np

# Example: distance of each node from the root in microns
tree.set_units("nm")
root = tree.get_root_coordinate()
coords = tree.get_node_coordinates()
dist_nm = np.linalg.norm(coords - root, axis=1)
tree.set_property("dist_from_root", dist_nm, level="v", create=True, dtype="double")
print(tree.get_property("dist_from_root")[:5])


[0.         0.16768326 0.346392   0.68085983 0.61251662]


## Persist custom properties in `.nr`


In [3]:
out = Path(tempfile.mkdtemp())
tree.save_tree(out / f"{tree.ID}.nr")
reloaded = nr.load(out / f"{tree.ID}.nr")
print(reloaded.has_property("dist_from_root", level="v"))
print(reloaded.get_property("dist_from_root")[:5])


True
[0.         0.16768326 0.346392   0.68085983 0.61251662]


## Functional API + `Forest.apply`


In [4]:
def mean_radius(t):
    return float(t.get_property("radius").mean())

radii = forest.apply(mean_radius, parallel=False)
print(dict(zip(forest.ids(), radii)))


{720575940596125868: 0.18437904732468033, 720575940599459782: 0.20187395745011852, 720575940599704006: 0.20064725570090508, 720575940599729862: 0.19109086308092854}


To add a **library** operation, implement graph logic in `utils/`, wrap in
`ops/tree_graphs/`, bind on {class}`~neurosetta.api.Tree`, and optionally re-export
from `neurosetta.__init__`. See {doc}`architecture`.

Next: {doc}`../api/index`.
